# Ch01-08: 데이터 품질 프레임워크


## 학습 목표

- 데이터 드리프트 탐지의 통계적 방법(PSI, KS, χ²)을 이해한다
- 스키마 검증 파이프라인을 설계한다
- 통계적 프로파일링으로 데이터 이상을 자동 탐지한다
- 데이터 품질 모니터링 대시보드를 구현한다


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 이론적 배경


### 1. Population Stability Index (PSI)

분포 변화 측정:

$$\text{PSI} = \sum_{i=1}^B \left(p_i^{\text{new}} - p_i^{\text{ref}}\right)\ln\frac{p_i^{\text{new}}}{p_i^{\text{ref}}}$$

| PSI | 해석 |
|-----|------|
| < 0.1 | 안정 |
| 0.1-0.25 | 주의 |
| > 0.25 | 유의한 변화 |

### 2. Kolmogorov-Smirnov 검정

$$D = \sup_x |F_{\text{ref}}(x) - F_{\text{new}}(x)|$$

### 3. 스키마 검증

- 타입 일관성, 범위 검사, 유일성 제약
- 참조 무결성, 분포 제약

### 4. 통계적 프로파일링

각 변수의 요약 통계량 + 이상 탐지: Z-score, IQR, 엔트로피 변화


## 구현 가이드

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# 참조 데이터 vs 새 데이터 (드리프트 시뮬레이션)
ref_data = np.random.normal(50, 10, 10000)
new_data = np.random.normal(52, 12, 10000)  # 약간의 드리프트

def compute_psi(ref, new, bins=10):
    breakpoints = np.percentile(ref, np.linspace(0,100,bins+1))
    breakpoints[0] = -np.inf; breakpoints[-1] = np.inf
    ref_counts = np.histogram(ref, breakpoints)[0] / len(ref)
    new_counts = np.histogram(new, breakpoints)[0] / len(new)
    ref_counts = np.clip(ref_counts, 1e-6, None)
    new_counts = np.clip(new_counts, 1e-6, None)
    return np.sum((new_counts-ref_counts)*np.log(new_counts/ref_counts))

psi = compute_psi(ref_data, new_data)
ks_stat, ks_p = stats.ks_2samp(ref_data, new_data)

print(f"PSI: {psi:.4f} ({'안정' if psi<0.1 else '주의' if psi<0.25 else '유의한 변화'})")
print(f"KS: stat={ks_stat:.4f}, p={ks_p:.6f}")


---
## 연습 문제

### 문제 1 [★]

다양한 드리프트 유형(평균 이동, 분산 변화, 분포 변화)에서 PSI, KS, χ² 검정의 민감도를 비교하라.

In [ ]:
# TODO: 드리프트 유형별 검정


### 문제 2 [★★]

범용 스키마 검증 클래스를 구현하라. 타입/범위/유일성/분포 제약을 선언적으로 정의하고 자동 검증.

In [ ]:
class SchemaValidator:
    # TODO
    pass


### 문제 3 [★★]

통계적 프로파일러를 구현하여 시계열 데이터에서 분포 이상을 자동 탐지하라. 윈도우별 통계량 변화를 모니터링.

In [ ]:
class DataProfiler:
    # TODO
    pass


### 문제 4 [★★★]

다변량 드리프트 탐지기를 설계하라. MMD(Maximum Mean Discrepancy) 또는 LSDD(Least-Squares Density Difference) 기반으로 고차원 분포 변화를 탐지하고, 부트스트랩 p-value를 계산.

In [ ]:
def mmd_test(X_ref, X_new, kernel='rbf', n_permutations=1000):
    # TODO
    pass


---
## 참고 자료

- Rabanser et al. (2019). Failing Loudly: An Empirical Study of Methods for Detecting Dataset Shift.
- Gretton et al. (2012). A Kernel Two-Sample Test. JMLR.
